# Notebook 03: Sentinel-1 Synthetic Aperture Radar (SAR) Acquisition
**GlobalFlood AI: Global Multimodal Flood Prediction System (48-Hour Forecast)**

### Objective
Understand synthetic aperture radar (SAR) physics for cloud-penetrating flood mapping:
1. C-band (5.405 GHz) microwave signal propagation through cloud cover
2. VV and VH dual-polarization backscatter characteristics
3. Decibel conversion ($\sigma^0_{	ext{dB}} = 10 \cdot \log_{10} \sigma^0$) and normalization to $[0.0, 1.0]$
4. Handling missing SAR imagery when a coordinate lacks recent scene acquisition (12-day repeat orbit)

### Inputs & Outputs
- **Inputs:** Coordinates and target acquisition timestamps
- **Outputs:** Preprocessed $128 	imes 128 	imes 2$ SAR patches and binary presence indicators
- **Dependencies:** `numpy`, `torch`, `ml.src.preprocessing`


In [ ]:
import numpy as np
from ml.src.preprocessing import MultimodalPreprocessor

# Demonstration of radar backscatter physics:
# Open water surfaces act like specular mirrors: incoming microwaves reflect away from sensor.
# Radar backscatter for water is very low: -25 dB to -18 dB.
# Rough ground, vegetation, and urban structures scatter microwaves diffusely: -14 dB to -6 dB.

water_sar = np.random.normal(loc=-22.0, scale=2.5, size=(2, 128, 128)).astype(np.float32)
land_sar = np.random.normal(loc=-12.0, scale=3.0, size=(2, 128, 128)).astype(np.float32)

print(f"Flooded patch mean dB: VV={water_sar[0].mean():.2f} dB, VH={water_sar[1].mean():.2f} dB")
print(f"Dry land patch mean dB: VV={land_sar[0].mean():.2f} dB, VH={land_sar[1].mean():.2f} dB")


### Preprocessing: Decibel Clipping & Normalization
Clip radar backscatter between $[-30	ext{ dB}, 0	ext{ dB}]$ and normalize linearly to $[0.0, 1.0]$.


In [ ]:
norm_water, avail_water = MultimodalPreprocessor.preprocess_sar_patch(water_sar)
norm_land, avail_land = MultimodalPreprocessor.preprocess_sar_patch(land_sar)

print("Normalized Water Patch range:", norm_water.min(), norm_water.max(), f"Availability: {avail_water}")
print("Normalized Land Patch range: ", norm_land.min(), norm_land.max(), f"Availability: {avail_land}")
assert norm_water.shape == (2, 128, 128)
assert norm_land.shape == (2, 128, 128)


### Handling Missing SAR Imagery
When Sentinel-1 has no scene within the orbital revisit window, the preprocessor outputs a zero tensor with `availability_flag = 0.0`.


In [ ]:
missing_patch, avail_missing = MultimodalPreprocessor.preprocess_sar_patch(None)
print("Missing SAR Patch shape:", missing_patch.shape)
print("Missing SAR Availability Flag:", avail_missing)
assert avail_missing == 0.0
assert np.all(missing_patch == 0.0)
print("\n[SUCCESS] SAR preprocessing and missing-data mechanics verified.")
